# Rede neural com atualização após cada exemplo

Construímos uma rede com uma camada oculta, sigmoide e uma saída binária. Usamos erro quadrático e retropropagação. Nesta versão, atualizamos os pesos depois de cada exemplo, seguindo a sequência apresentada na aula 6, página 17.

A versão anterior fez a atualização pela média dos gradientes da época. Mantemos aquele experimento no notebook 01 e fazemos esta comparação separadamente. Aqui a rede é implementada do zero; usamos NumPy para as operações numéricas.

O arquivo para o Kaggle será gerado ao final, após fixarmos a configuração e treinarmos a rede com todas as 520 amostras rotuladas.

### Resultado desta execução

A rede escolhida tem **4 entradas, 12 neurônios ocultos e uma saída**, com 73 pesos ajustáveis, incluindo os vieses.

| Escolha | Configuração |
|---|---|
| Atributos | SYSTOLIC, GLUCOSE, HEMOGLOBIN, TRIGLYCERIDES |
| Valores numéricos ausentes | Mediana calculada no treinamento |
| Normalização | Média e desvio padrão calculados no treinamento |
| Taxa de aprendizado por exemplo | 0.1 |
| Épocas | 800 |
| Inicialização dos pesos e embaralhamento | Semente 7 |
| Aprendizado | Erro quadrático e retropropagação, atualização após cada exemplo |
| Decisão | Saída maior que 0,5 indica diabetes |

| Avaliação | Acurácia | F1 macro |
|---|---|---|
| Escolha: divisão 42, pesos 7 | 88.70% | 88.70% |
| Conferência: divisão 73, pesos 7 | 88.71% | 88.67% |
| Conferência: divisão 101, pesos 7 | 88.22% | 88.20% |
| Conferência: divisão 42, pesos 19 | 88.22% | 88.20% |
| Conferência: divisão 42, pesos 31 | 88.46% | 88.45% |
| 104 amostras reservadas | 85.58% | 85.58% |

Escolhemos a configuração entre 54 possibilidades iniciais e mais duas extensões de épocas, pelo F1 macro médio na divisão 42. As demais divisões e inicializações conferem essa escolha, usando as mesmas pessoas em outras condições de treinamento. Não alteramos a configuração para melhorar essas conferências ou a reserva.

Na reserva, acertamos 89 de 104 amostras. Como essa reserva já foi observada no projeto, usamos o resultado como comparação local. O desempenho no Kaggle pode ser diferente.

O arquivo para envio desta versão é `resultados/submissao_rede_neural_exemplos.csv`. Ele contém 312 previsões produzidas pela rede treinada novamente com todas as 520 amostras rotuladas. O envio ao Kaggle ainda precisa ser feito.

### Base e avaliação

Mantemos 416 amostras para escolher a configuração e as mesmas 104 reservadas dos outros modelos. Cada paciente aparece uma única vez na base padronizada.

Escolhemos a configuração pelo F1 macro médio em cinco partes, usando a divisão de semente 42. A inicialização dos pesos e a ordem dos exemplos usam a semente 7. Depois conferimos a escolha em outras divisões e inicializações. O teste do Kaggle recebe apenas as previsões finais.

As 104 amostras já foram consultadas durante o projeto. Elas permitem uma comparação local entre os modelos, mas não representam uma avaliação externa nova.

In [1]:
from pathlib import Path
from itertools import product
from time import perf_counter
import csv
import hashlib
import json
import random
import sys

import numpy as np

raiz = Path.cwd()
if not (raiz / "dados" / "treinamento_padronizado.csv").is_file():
    raiz = raiz.parent
caminho_base = raiz / "dados" / "treinamento_padronizado.csv"
with caminho_base.open(encoding="utf-8-sig", newline="") as arquivo:
    amostras = list(csv.DictReader(arquivo))

grupos = {
    "Três exames": ("GLUCOSE", "HEMOGLOBIN", "TRIGLYCERIDES"),
    "Quatro medidas": ("SYSTOLIC", "GLUCOSE", "HEMOGLOBIN", "TRIGLYCERIDES"),
}
classes = np.array([int(a["class"]) for a in amostras], dtype=np.int8)
pacientes = [a["PATIENT_CODE"].strip() for a in amostras]
if any(not p for p in pacientes) or len(set(pacientes)) != len(pacientes):
    raise ValueError("Conferir os registros por paciente.")
if not np.all(np.isin(classes, [0, 1])):
    raise ValueError("Esperávamos as classes 0 e 1.")

controle = np.flatnonzero(classes == 0).tolist()
diabetes = np.flatnonzero(classes == 1).tolist()
gerador = random.Random(42)
gerador.shuffle(controle)
gerador.shuffle(diabetes)
n0, n1 = round(len(controle) * 0.20), round(len(diabetes) * 0.20)
reservadas = np.array(controle[:n0] + diabetes[:n1])
treinamento = np.array(controle[n0:] + diabetes[n1:])


def dividir(indices, semente):
    partes = [[] for _ in range(5)]
    gerador = random.Random(semente)
    for classe in (0, 1):
        grupo = [int(i) for i in indices if classes[i] == classe]
        gerador.shuffle(grupo)
        for posicao, i in enumerate(grupo):
            partes[posicao % 5].append(i)
    for posicao, parte in enumerate(partes):
        referencia = [i for j, outra in enumerate(partes) if j != posicao for i in outra]
        yield np.array(referencia), np.array(parte)


print(f"Python {sys.version.split()[0]} | NumPy {np.__version__}")
print(f"Treinamento: {len(treinamento)} | Reservadas: {len(reservadas)}")
print("SHA-256:", hashlib.sha256(caminho_base.read_bytes()).hexdigest())

Python 3.12.14 | NumPy 2.3.5
Treinamento: 416 | Reservadas: 104
SHA-256: bc1a686d83c06e2d7667a6089e82b54b4bff2ca6467b81155f1246ac4719a032


### Preparação das entradas

Partimos de dois grupos pequenos que se mostraram úteis nos modelos anteriores: glicose, hemoglobina e triglicerídeos; ou esses três exames mais a pressão sistólica. Essa seleção é uma escolha deste experimento, e não uma afirmação de que os demais atributos sejam irrelevantes.

Preenchemos os vazios numéricos com a mediana e normalizamos pela média e desvio padrão. Essas escolhas vieram da exploração anterior da rede. Em cada divisão, os valores usados para preencher e normalizar são calculados apenas nos exemplos usados para treinar.

Mantemos a premissa anterior sobre HbA1c: `HEMOGLOBIN` maior que 100 é tratada como ausente. Valores não positivos nas medidas também são tratados como ausentes. A regra é aplicada igualmente no treinamento, na avaliação e no teste.

In [2]:
def ler_numero(amostra, coluna):
    try:
        valor = float(str(amostra[coluna]).strip().replace(",", "."))
    except ValueError:
        return np.nan
    if not np.isfinite(valor) or valor <= 0:
        return np.nan
    if coluna == "HEMOGLOBIN" and valor > 100:
        return np.nan
    return valor


def ajustar_preparo(referencia, grupo, escala, preenchimento):
    parametros = {"grupo": grupo, "escala": escala, "preenchimento": preenchimento, "numeros": {}}
    for coluna in grupos[grupo]:
        valores = np.array([ler_numero(a, coluna) for a in referencia])
        disponiveis = valores[np.isfinite(valores)]
        if not len(disponiveis):
            raise ValueError(f"Nenhuma medida disponível para {coluna}.")
        estimativa = np.mean(disponiveis) if preenchimento == "media" else np.median(disponiveis)
        if escala == "minmax":
            centro = np.min(disponiveis)
            divisor = np.ptp(disponiveis)
        else:
            centro = np.mean(disponiveis)
            divisor = np.std(disponiveis, ddof=1) if len(disponiveis) > 1 else 0
        parametros["numeros"][coluna] = {
            "estimativa": float(estimativa), "centro": float(centro), "divisor": float(divisor),
        }
    parametros["entradas"] = list(grupos[grupo])
    return parametros


def transformar(consulta, parametros):
    linhas = []
    for amostra in consulta:
        linha = []
        for coluna in grupos[parametros["grupo"]]:
            regra = parametros["numeros"][coluna]
            valor = ler_numero(amostra, coluna)
            if not np.isfinite(valor):
                valor = regra["estimativa"]
            normalizado = (valor - regra["centro"]) / regra["divisor"] if regra["divisor"] else 0.0
            linha.append(normalizado)
        linhas.append(linha)
    return np.array(linhas, dtype=float)


### Como a rede calcula uma resposta

Cada neurônio oculto multiplica as entradas pelos seus pesos, soma os resultados e aplica a sigmoide. A saída recebe os resultados desses neurônios e faz outra soma ponderada, seguida de outra sigmoide.

O resultado fica entre 0 e 1. Acima de 0,5 prevemos diabetes; caso contrário, controle. Esse resultado não foi calibrado como probabilidade clínica.

O viés permite deslocar a resposta do neurônio. Usamos uma entrada fixa igual a -1, seguindo a convenção da aula. Os pesos começam com valores pequenos e aleatórios; a semente permite repetir o começo do treinamento.

In [3]:
def sigmoide(z):
    resultado = np.empty_like(z, dtype=float)
    positivos = z >= 0
    resultado[positivos] = 1 / (1 + np.exp(-z[positivos]))
    exponencial = np.exp(z[~positivos])
    resultado[~positivos] = exponencial / (1 + exponencial)
    return resultado


def incluir_vies(x):
    return np.column_stack((x, -np.ones(len(x))))


def passar(x_com_vies, pesos_entrada, pesos_saida):
    oculta = sigmoide(x_com_vies @ pesos_entrada)
    oculta_com_vies = incluir_vies(oculta)
    saida = sigmoide(oculta_com_vies @ pesos_saida)
    return oculta, oculta_com_vies, saida


def prever(x, modelo):
    _, _, saida = passar(incluir_vies(x), modelo[0], modelo[1])
    return (saida.ravel() > 0.5).astype(np.int8)

### Como os pesos aprendem

Para cada exemplo, fazemos a previsão, comparamos a resposta com a classe conhecida e calculamos o ajuste pela retropropagação. Primeiro calculamos os ajustes da saída e da camada oculta usando os pesos atuais. Depois alteramos os pesos.

A taxa de aprendizado controla o tamanho de cada ajuste. Uma época corresponde a uma passagem por todos os exemplos de treinamento. Embaralhamos a ordem dos exemplos no começo de cada época, de forma reproduzível.

Guardamos os pesos nas épocas que queremos comparar. Não treinamos com as respostas da avaliação. Uma rede pode continuar reduzindo o erro do treinamento e piorar em exemplos novos; por isso a escolha é feita na validação.

In [4]:
def treinar_exemplos(x, y, ocultos, taxa, pontos, semente=7, registrar=False):
    gerador = np.random.default_rng(semente)
    pesos_entrada = gerador.uniform(-0.5, 0.5, size=(x.shape[1] + 1, ocultos))
    pesos_saida = gerador.uniform(-0.5, 0.5, size=ocultos + 1)
    x_com_vies = incluir_vies(x)
    y = np.asarray(y, dtype=float)
    oculta_com_vies = np.empty(ocultos + 1)
    oculta_com_vies[-1] = -1
    modelos = {}
    historico = []

    with np.errstate(over="ignore"):
        for epoca in range(1, max(pontos) + 1):
            for i in gerador.permutation(len(x)):
                oculta = 1 / (1 + np.exp(-(x_com_vies[i] @ pesos_entrada)))
                oculta_com_vies[:-1] = oculta
                saida = 1 / (1 + np.exp(-(oculta_com_vies @ pesos_saida)))
                delta_saida = (y[i] - saida) * saida * (1 - saida)
                delta_oculta = delta_saida * pesos_saida[:-1] * oculta * (1 - oculta)
                pesos_entrada += taxa * np.outer(x_com_vies[i], delta_oculta)
                pesos_saida += taxa * oculta_com_vies * delta_saida

            if epoca in pontos or (registrar and epoca % 20 == 0):
                modelo = (pesos_entrada.copy(), pesos_saida.reshape(-1, 1).copy())
                _, _, respostas = passar(x_com_vies, *modelo)
                erro = float(np.mean((y - respostas.ravel()) ** 2) / 2)
                if not np.isfinite(erro) or not all(np.isfinite(p).all() for p in modelo):
                    raise ValueError("O treinamento produziu pesos ou erro inválidos.")
                if registrar:
                    historico.append((epoca, erro))
                if epoca in pontos:
                    modelos[epoca] = modelo
    return modelos, historico

#### Como contamos os acertos e erros

A matriz de confusão separa os acertos e erros de cada classe. Registramos acurácia, precisão e recall, além do F1 macro usado para comparar as configurações. O F1 macro é a média do F1 de controle e do F1 de diabetes.

In [5]:
def contar(reais, previstas):
    return np.array([
        np.sum((reais == 0) & (previstas == 0)),
        np.sum((reais == 0) & (previstas == 1)),
        np.sum((reais == 1) & (previstas == 0)),
        np.sum((reais == 1) & (previstas == 1)),
    ], dtype=np.int16)


def razao(a, b):
    return np.divide(a, b, out=np.zeros_like(a, dtype=float), where=b != 0)


def metricas(contagens):
    tn, fp, fn, tp = np.moveaxis(contagens, -1, 0)
    return {
        "acuracia": razao(tn + tp, tn + fp + fn + tp),
        "f1": (razao(2 * tn, 2 * tn + fp + fn) + razao(2 * tp, 2 * tp + fp + fn)) / 2,
        "precisao": (razao(tn, tn + fn) + razao(tp, tp + fp)) / 2,
        "precisao0": razao(tn, tn + fn), "precisao1": razao(tp, tp + fp),
        "recall0": razao(tn, tn + fp), "recall1": razao(tp, tp + fn),
    }


def mostrar(nome, contagens):
    m = metricas(contagens)
    print(f"{nome:25} | " + " | ".join(
        f"{np.mean(m[c]):>8.2%}" for c in ("acuracia", "f1", "precisao", "recall0", "recall1")
    ))

### Comparação das configurações

Comparamos os dois grupos de atributos, 4, 8 ou 12 neurônios ocultos, taxas de aprendizado 0,01, 0,05 ou 0,1 e 50, 150 ou 400 épocas. São 54 configurações iniciais. Cada configuração passa pelas mesmas cinco partes da validação.

Uma trajetória de treinamento permite observar três quantidades de épocas. Escolhemos pelo maior F1 macro médio; em caso de empate, preferimos menos atributos, menos neurônios e menos épocas. A inicialização 7 permanece fixa.

In [6]:
grupos_comparados = ("Três exames", "Quatro medidas")
trajetos = list(product(grupos_comparados, (4, 8, 12), (0.01, 0.05, 0.1)))
pontos = (50, 150, 400)
configuracoes = [trajeto + (epoca,) for trajeto in trajetos for epoca in pontos]
rodadas = []
inicio = perf_counter()


def preparar_divisao(indices_ref, indices_aval, grupo):
    ref = [amostras[i] for i in indices_ref]
    aval = [amostras[i] for i in indices_aval]
    preparo = ajustar_preparo(ref, grupo, "desvio", "mediana")
    return transformar(ref, preparo), transformar(aval, preparo), preparo


for numero, (ref, aval) in enumerate(dividir(treinamento, 42), start=1):
    cache = {}
    resultados = []
    for grupo, ocultos, taxa in trajetos:
        if grupo not in cache:
            cache[grupo] = preparar_divisao(ref, aval, grupo)
        x, consulta, _ = cache[grupo]
        modelos, _ = treinar_exemplos(x, classes[ref], ocultos, taxa, pontos)
        for epoca in pontos:
            resultados.append(contar(classes[aval], prever(consulta, modelos[epoca])))
    rodadas.append(resultados)
    print(f"Rodada {numero} concluída", flush=True)

rodadas = np.array(rodadas)
f1_medios = metricas(rodadas)["f1"].mean(axis=0)
ordem = sorted(range(len(configuracoes)), key=lambda i: (
    -f1_medios[i], len(grupos[configuracoes[i][0]]), configuracoes[i][1],
    configuracoes[i][3], configuracoes[i][2], i,
))
escolhida = ordem[0]
configuracao_final = configuracoes[escolhida]
print(f"Tempo da busca: {perf_counter() - inicio:.1f} s")
print("Configurações:", len(configuracoes))
for i in ordem[:10]:
    print(f"F1 {f1_medios[i]:.2%} | {configuracoes[i]}")
print("Escolhida:", configuracao_final)

Rodada 1 concluída
Rodada 2 concluída
Rodada 3 concluída
Rodada 4 concluída
Rodada 5 concluída
Tempo da busca: 192.3 s
Configurações: 54
F1 87.98% | ('Quatro medidas', 12, 0.1, 400)
F1 86.27% | ('Quatro medidas', 8, 0.1, 400)
F1 85.30% | ('Três exames', 12, 0.1, 400)
F1 84.61% | ('Três exames', 8, 0.1, 400)
F1 82.67% | ('Quatro medidas', 12, 0.05, 400)
F1 81.71% | ('Quatro medidas', 8, 0.05, 400)
F1 81.67% | ('Três exames', 4, 0.1, 400)
F1 80.52% | ('Quatro medidas', 4, 0.1, 400)
F1 79.72% | ('Quatro medidas', 8, 0.1, 150)
F1 79.34% | ('Quatro medidas', 12, 0.1, 150)
Escolhida: ('Quatro medidas', 12, 0.1, 400)


#### Mais tempo de aprendizado para a candidata escolhida

A melhor configuração da busca inicial usa 400 épocas, o maior tempo de treinamento que havíamos comparado. Para conferir esse limite, repetimos essa mesma trajetória com 800 e 1.600 épocas, mantendo os atributos, os neurônios e a taxa. Acrescentamos duas configurações à comparação: são 56 no total.

A escolha continua sendo pelo F1 macro médio nas cinco partes da divisão 42. A reserva não define a quantidade de épocas. Preservamos o registro da primeira avaliação em `resultados/rede_neural_exemplos_busca_inicial.npz`; naquela etapa, a rede acertou 90 de 104 amostras reservadas.

In [7]:
trajeto_estendido = configuracao_final[:3]
pontos_adicionais = (800, 1600)
rodadas_adicionais = []
for numero, (ref, aval) in enumerate(dividir(treinamento, 42), start=1):
    grupo, ocultos, taxa = trajeto_estendido
    x, consulta, _ = preparar_divisao(ref, aval, grupo)
    modelos, _ = treinar_exemplos(x, classes[ref], ocultos, taxa, pontos_adicionais)
    rodadas_adicionais.append([
        contar(classes[aval], prever(consulta, modelos[epoca])) for epoca in pontos_adicionais
    ])
    print(f"Extensão, rodada {numero} concluída", flush=True)

configuracoes += [trajeto_estendido + (epoca,) for epoca in pontos_adicionais]
rodadas = np.concatenate((rodadas, np.array(rodadas_adicionais)), axis=1)
f1_medios = metricas(rodadas)["f1"].mean(axis=0)
ordem = sorted(range(len(configuracoes)), key=lambda i: (
    -f1_medios[i], len(grupos[configuracoes[i][0]]), configuracoes[i][1],
    configuracoes[i][3], configuracoes[i][2], i,
))
escolhida = ordem[0]
configuracao_final = configuracoes[escolhida]
print("Escolhida após a extensão:", configuracao_final)
for i in ordem[:5]:
    print(f"F1 {f1_medios[i]:.2%} | {configuracoes[i]}")

Extensão, rodada 1 concluída
Extensão, rodada 2 concluída
Extensão, rodada 3 concluída
Extensão, rodada 4 concluída
Extensão, rodada 5 concluída
Escolhida após a extensão: ('Quatro medidas', 12, 0.1, 800)
F1 88.70% | ('Quatro medidas', 12, 0.1, 800)
F1 88.69% | ('Quatro medidas', 12, 0.1, 1600)
F1 87.98% | ('Quatro medidas', 12, 0.1, 400)
F1 86.27% | ('Quatro medidas', 8, 0.1, 400)
F1 85.30% | ('Três exames', 12, 0.1, 400)


### Conferências e comparação local

Fixamos os atributos, o número de neurônios, a taxa e as épocas. Conferimos a configuração nas divisões 73 e 101 com a inicialização 7, e na divisão 42 com inicializações 19 e 31. Mantemos a inicialização 7 no modelo final, sem escolher a semente que tiver melhor resultado.

Depois treinamos com as 416 amostras e avaliamos nas 104 reservadas. A tabela recupera os resultados dos outros modelos nessa mesma reserva, para facilitar a comparação. As conferências não mudam a configuração escolhida.

In [8]:
def ajustar_rede(ref, aval, semente=7, registrar=False):
    grupo, ocultos, taxa, epoca = configuracao_final
    x, consulta, preparo = preparar_divisao(ref, aval, grupo)
    modelos, historico = treinar_exemplos(x, classes[ref], ocultos, taxa, (epoca,), semente, registrar)
    modelo = modelos[epoca]
    return contar(classes[aval], prever(consulta, modelo)), modelo, preparo, historico


conferencias = {}
for divisao, pesos in ((73, 7), (101, 7), (42, 19), (42, 31)):
    conferencias[divisao, pesos] = np.array([
        ajustar_rede(ref, aval, pesos)[0] for ref, aval in dividir(treinamento, divisao)
    ])

print("Avaliação                 | Acurácia | F1 macro | Precisão | Recall 0 | Recall 1")
mostrar("Escolha, divisão 42", rodadas[:, escolhida])
for (divisao, pesos), contagens in conferencias.items():
    mostrar(f"Divisão {divisao}, pesos {pesos}", contagens)

reserva, modelo_reserva, preparo_reserva, historico_reserva = ajustar_rede(treinamento, reservadas, registrar=True)
print("Nas 104 amostras reservadas")
mostrar("Primeiro KNN enviado", np.array([44, 6, 8, 46]))
mostrar("Árvore quatro atributos", np.array([43, 7, 9, 45]))
mostrar("Mistura KNN e árvore", np.array([44, 6, 7, 47]))
mostrar("Rede, atualização em lote", np.array([43, 7, 10, 44]))
mostrar("Rede, exemplo a exemplo", reserva)
print("Matriz da rede:", reserva.reshape(2, 2).tolist())
print("Acertos:", int(reserva[0] + reserva[3]), "de", len(reservadas))

Avaliação                 | Acurácia | F1 macro | Precisão | Recall 0 | Recall 1
Escolha, divisão 42       |   88.70% |   88.70% |   88.77% |   90.50% |   87.04%
Divisão 73, pesos 7       |   88.71% |   88.67% |   89.20% |   90.00% |   87.54%
Divisão 101, pesos 7      |   88.22% |   88.20% |   88.32% |   88.00% |   88.45%
Divisão 42, pesos 19      |   88.22% |   88.20% |   88.34% |   89.50% |   87.02%
Divisão 42, pesos 31      |   88.46% |   88.45% |   88.46% |   89.50% |   87.51%
Nas 104 amostras reservadas
Primeiro KNN enviado      |   86.54% |   86.53% |   86.54% |   88.00% |   85.19%
Árvore quatro atributos   |   84.62% |   84.61% |   84.62% |   86.00% |   83.33%
Mistura KNN e árvore      |   87.50% |   87.49% |   87.48% |   88.00% |   87.04%
Rede, atualização em lote |   83.65% |   83.65% |   83.70% |   86.00% |   81.48%
Rede, exemplo a exemplo   |   85.58% |   85.58% |   85.63% |   88.00% |   83.33%
Matriz da rede: [[44, 6], [9, 45]]
Acertos: 89 de 104


### Arquitetura e aprendizado

O número de entradas é a quantidade de atributos usados. A quantidade de neurônios ocultos veio da validação. Há um único neurônio de saída. Os vieses estão incluídos na contagem de pesos ajustáveis.

A tabela mostra o erro quadrático nos exemplos de treinamento. Esse erro acompanha os ajustes dos pesos; ele é diferente do F1 macro e da acurácia medidos na avaliação.

In [9]:
print(f"Arquitetura: {len(preparo_reserva['entradas'])} entradas -> {configuracao_final[1]} neurônios ocultos -> 1 saída")
print("Atributos:", ", ".join(preparo_reserva["entradas"]))
print("Pesos ajustáveis:", sum(p.size for p in modelo_reserva))
print("Época | Erro quadrático de treinamento")
for epoca, erro in historico_reserva[::max(1, len(historico_reserva) // 10)]:
    print(f"{epoca:>5} | {erro:.6f}")
print("Última época:", historico_reserva[-1])

caminho_busca = raiz / "resultados" / "rede_neural_exemplos_busca.npz"
np.savez_compressed(
    caminho_busca,
    catalogo=json.dumps({
        "configuracoes": configuracoes, "grupos": grupos,
        "conferencias": list(conferencias), "normalizacao": "desvio", "preenchimento": "mediana",
        "semente_divisao": 42, "semente_modelo": 7, "atualizacao": "exemplo a exemplo",
        "sha256_base": hashlib.sha256(caminho_base.read_bytes()).hexdigest(),
    }, ensure_ascii=False),
    busca=rodadas, escolhida=escolhida, conferencias=np.array(list(conferencias.values())),
    reserva=reserva, historico_reserva=np.array(historico_reserva),
)
print("Registro:", caminho_busca.relative_to(raiz))

Arquitetura: 4 entradas -> 12 neurônios ocultos -> 1 saída
Atributos: SYSTOLIC, GLUCOSE, HEMOGLOBIN, TRIGLYCERIDES
Pesos ajustáveis: 73
Época | Erro quadrático de treinamento
   20 | 0.084705
  100 | 0.069480
  180 | 0.053747
  260 | 0.037436
  340 | 0.033378
  420 | 0.030149
  500 | 0.028409
  580 | 0.026197
  660 | 0.025178
  740 | 0.025540
Última época: (800, 0.024827329008184178)
Registro: resultados\rede_neural_exemplos_busca.npz


### Treinamento final e submissão

Treinamos novamente, agora com as 520 amostras, usando a configuração e a inicialização fixadas. A mediana, a média e o desvio padrão são recalculados nesse treinamento completo. Aplicamos esses parâmetros às 312 linhas do teste para gerar as previsões.

Salvamos os pesos e a preparação dos atributos. Reabrimos o modelo salvo e conferimos se ele produz as mesmas previsões. O CSV tem apenas as colunas `id` e `class`, preservando a ordem e os identificadores do teste.

In [10]:
def padronizar_teste(amostra):
    entrada = amostra.copy()
    pressao = entrada["BLOOD_PRESSURE"].strip().replace("x", "/").replace(" ", "")
    if pressao in {"", "?"}:
        entrada["SYSTOLIC"] = ""
    else:
        sistolica, _ = pressao.split("/")
        entrada["SYSTOLIC"] = str(int(sistolica))
    return entrada


with (raiz / "dados" / "kaggle-test" / "test.csv").open(encoding="utf-8-sig", newline="") as arquivo:
    teste_original = list(csv.DictReader(arquivo))
ids_teste = [a["id"] for a in teste_original]
if not ids_teste or len(set(ids_teste)) != len(ids_teste) or any(not i.strip() for i in ids_teste):
    raise ValueError("O teste precisa ter IDs únicos e preenchidos.")
teste = [padronizar_teste(a) for a in teste_original]

grupo, ocultos, taxa, epoca = configuracao_final
escala, preenchimento = "desvio", "mediana"
preparo_final = ajustar_preparo(amostras, grupo, escala, preenchimento)
x_final = transformar(amostras, preparo_final)
x_teste = transformar(teste, preparo_final)
modelos_finais, historico_final = treinar_exemplos(
    x_final, classes, ocultos, taxa, (epoca,), semente=7, registrar=True
)
modelo_final = modelos_finais[epoca]
previsoes_teste = prever(x_teste, modelo_final)

caminho_modelo = raiz / "resultados" / "rede_neural_exemplos_modelo.npz"
np.savez_compressed(
    caminho_modelo, pesos_entrada=modelo_final[0], pesos_saida=modelo_final[1],
    preparo=json.dumps(preparo_final, ensure_ascii=False),
    configuracao=json.dumps(configuracao_final, ensure_ascii=False),
    semente_pesos=7,
    sha256_base=hashlib.sha256(caminho_base.read_bytes()).hexdigest(),
)
caminho_envio = raiz / "resultados" / "submissao_rede_neural_exemplos.csv"
with caminho_envio.open("w", encoding="utf-8", newline="") as arquivo:
    escritor = csv.writer(arquivo)
    escritor.writerow(["id", "class"])
    escritor.writerows(zip(ids_teste, previsoes_teste.tolist(), strict=True))

with np.load(caminho_modelo, allow_pickle=False) as salvo:
    preparo_carregado = json.loads(salvo["preparo"].item())
    modelo_carregado = (salvo["pesos_entrada"], salvo["pesos_saida"])
    previsoes_carregadas = prever(transformar(teste, preparo_carregado), modelo_carregado)
if not np.array_equal(previsoes_teste, previsoes_carregadas):
    raise ValueError("O modelo salvo não reproduziu as previsões.")

print("Modelo treinado com:", len(amostras), "amostras rotuladas")
print("Modelo salvo:", caminho_modelo.relative_to(raiz))
print("CSV salvo:", caminho_envio.relative_to(raiz))
print("Previsões:", len(previsoes_teste))
print("SHA-256 do CSV:", hashlib.sha256(caminho_envio.read_bytes()).hexdigest())

Modelo treinado com: 520 amostras rotuladas
Modelo salvo: resultados\rede_neural_exemplos_modelo.npz
CSV salvo: resultados\submissao_rede_neural_exemplos.csv
Previsões: 312
SHA-256 do CSV: b72014129dd5c6c2a08b22a0d2fe0f31073e8ce3ef58971ef0e2954ccdf7d9fd


### Referências

- [Aula 6](../docs/materiais_aula/aula6_MD.pdf): soma ponderada e viés, página 6; sigmoide, 7; arquitetura, 9, 10 e 12; saída binária, 11; erro quadrático e gradiente, 15 e 16; atualização exemplo a exemplo e épocas, 17; escolha de neurônios por validação cruzada, 19; retropropagação, 21 a 23.
- [Aula 2](../docs/materiais_aula/aula2_MD.pdf): valores ausentes e seleção de atributos.
- [Aula 4](../docs/materiais_aula/aula4_MD.pdf): média e mediana, página 9.
- [Aula 7](../docs/materiais_aula/aula7_MD.pdf): holdout e validação cruzada, páginas 7 e 8; métricas, 10 e 11.
- [Aula 10](../docs/materiais_aula/aula10_MD.pdf): normalização pela média e desvio padrão, página 16.

As páginas são as posições no PDF, contando a capa. Os grupos de atributos, a grade de configurações, as sementes e o embaralhamento por época são escolhas desta aplicação dos conceitos.